In [1]:
# pip install "transformers>=4.43" datasets tokenizers accelerate evaluate sentencepiece
from pathlib import Path
from datetime import datetime
import math, itertools

import numpy as np
from datasets import load_dataset
import torch

from transformers import set_seed
set_seed(42)
from transformers import AutoTokenizer, LlamaConfig, LlamaForCausalLM, TrainingArguments, Trainer
from transformers import DataCollatorForLanguageModeling, TrainerCallback, EarlyStoppingCallback

In [2]:
def now():
    return datetime.now().astimezone().strftime('%FT%T%:z')

class TsvLogger(TrainerCallback):
    def __init__(self, path=Path("logs") / "training.tsv"):
        path.parent.mkdir(parents=True, exist_ok=True)
        self.path = path

        columns = ['step', 'epoch', 'split', 'loss', 'learning_rate', 'created_at']
        if self.path.exists():
            return

        with open(self.path, "w", encoding="utf-8") as f:
            f.write('\t'.join(columns) + '\n')

    def on_log(self, args, state, control, logs=None, **kwargs):
        if not state.is_world_process_zero: # 分布式只在主进程写
            return

        if logs is None or logs.get("loss") is None:
            return

        #print(f"on_log: args={args}, state={state}, control={control}, logs={logs}")

        split = "train"
        step = logs.get("step", state.global_step)
        epoch = max(1, math.ceil(state.epoch))
        loss = round(logs.get("loss", np.nan), 3)        # 训练损失
        lr = round(logs.get("learning_rate", np.nan), 6) 

        values = [str(step), str(epoch), split, str(loss), str(lr), now()]

        with open(self.path, "a", encoding="utf-8") as f:
            f.write('\t'.join(values) + '\n')

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        if not state.is_world_process_zero:
            return

        if metrics is None or metrics.get("eval_loss") is None:
            return

        #print(f"on_evaluate: args={args}, state={state}, control={control}, metrics={metrics}")

        split = "eval"
        epoch = max(1, math.ceil(state.epoch))
        loss = round(metrics.get("eval_loss", np.nan), 3)
        lr = round(metrics.get("learning_rate", np.nan), 6)

        values = [str(state.global_step), str(epoch), split, str(loss), str(lr), now()]

        with open(self.path, "a", encoding="utf-8") as f:
            f.write('\t'.join(values) + '\n')

In [3]:
#### 1.
run_name = "ch13"

data_dir = Path("data") / "ch13"
#data_dir.mkdir(parents=True, exist_ok=True)

In [4]:
#### 2. tokenizer
tokenizer = AutoTokenizer.from_pretrained(data_dir / "tokenizer", trust_remote_code=True)
tokenizer.padding_side = "right"

# model parameters
m_params = {
    'vocab_size': tokenizer.vocab_size,
    'd_model': 512,
    'block_size': 1_024,
    'n_head': 8,
    'n_layer': 4,
    
    'max_position_embeddings': 2_048,
}

assert(m_params['d_model'] % m_params['n_head'] == 0)
assert(m_params['block_size'] <= m_params['max_position_embeddings'])
print(f"Model parameters: {m_params}")

Model parameters: {'vocab_size': 32000, 'd_model': 512, 'block_size': 1024, 'n_head': 8, 'n_layer': 4, 'max_position_embeddings': 2048}


In [5]:
#### 3. dataset
raw_ds = load_dataset("json", data_files={
    "train": str(data_dir / "train.jsonl"),
    "validation": str(data_dir / "val.jsonl"),
})
# jsonl: {"text": "Hello, world and 2025!"}\n{"text": "42"}....

def tokenize(split):
    return tokenizer(
        [t + tokenizer.eos_token for t in split["text"]],
        add_special_tokens=False,
        return_attention_mask=False,
        return_token_type_ids=False,
        # truncation=False
    )

def chunk(d):
    all_ids = list(itertools.chain.from_iterable(d["input_ids"]))
    bs = m_params['block_size']

    chunks = [all_ids[i:i+bs] for i in range(0, len(all_ids) - bs, bs)]

    return { "input_ids": chunks, "labels": chunks.copy() }

#from trl import ConstantLengthDataset

#packed = ConstantLengthDataset(
#    tokenizer=tok,
#    dataset=ds["train"],
#    seq_length=1024,
#    eos_token_id=tok.eos_token_id,
#    # formatting_func=lambda ex: ex["text"],  # 如果字段不是 'text' 就提供格式化函数
#)

tokd_ds = raw_ds.map(
    tokenize,
    batched=True,
    remove_columns=raw_ds["train"].column_names,
)
# tokenize every records(texts)
# tokd_ds['train'][0]: {'input_ids': [10023, 14030, 8954, 2885, 3675, 8597, 19817....]}

lm_ds = tokd_ds.map(
    chunk,
    batched=True,
    num_proc=4,
    remove_columns=tokd_ds["train"].column_names,  # 关键：删掉残留列，避免长度不一致
)
# len(lm_ds['train'][0]['input_ids']) == block_size
# len(lm_ds['train'][0]['labels']) == block_size

print(f"dataset: train={len(lm_ds['train'])}, validation={len(lm_ds['validation'])}")
print()
print(tokenizer.decode(lm_ds['train'][0]['input_ids'])[:300]) # multiply text separated by [EOS]

dataset: train=3879, validation=199

Can you write a short introduction about the relevance of the term "monopsony" in economics? Please use examples related to potential monopsonies in the labour market and cite relevant research.[EOS]"Monopsony" refers to a market structure where there is only one buyer for a particular good or servi


In [6]:
print(lm_ds)

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'labels'],
        num_rows: 3879
    })
    validation: Dataset({
        features: ['input_ids', 'labels'],
        num_rows: 199
    })
})


In [7]:
#tokens = lm_ds['train'][0]['input_ids']
#print(tokens)
#print(tokenizer.decode(tokens))

In [8]:
#### 4. model
cfg = LlamaConfig(
    vocab_size=m_params['vocab_size'],
    hidden_size=m_params['d_model'],                               # 小模型先跑通；大模型如 2048/4096
    intermediate_size=int(m_params['d_model']*2.66),               # ~ hidden_size*2.66（SwiGLU比例）
    num_hidden_layers=m_params['n_layer'],                         # 层数
    num_attention_heads=m_params['n_head'],                        # 注意力头

    max_position_embeddings=m_params['max_position_embeddings'],   # 上下文长度
    rms_norm_eps=1e-5,
    rope_theta=10000.0,                                            # RoPE 频率
    #rope_scaling={"type": "linear", "factor": 4},
    pad_token_id=tokenizer.pad_token_id,
)

args = TrainingArguments(
    output_dir=data_dir,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=8, # 16,
    gradient_checkpointing=True,
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    weight_decay=0.1,
    logging_steps=50,
    max_grad_norm=1.0,
    report_to="none",

    learning_rate=3e-4,
    lr_scheduler_type="reduce_lr_on_plateau", # or "consine"
    max_steps=20_000,                         # or num_train_epochs=200
    eval_steps=200, # 500,
    save_steps=200, # 500,
    #warmup_steps=1_000,
    #warmup_ratio=0.0,
    #warmup_ratio=0.03,                       # ignored when lr_scheduler_type="reduce_lr_on_plateau"

    restore_callback_states_from_checkpoint=True,
    eval_strategy="steps", # or epoch
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    load_best_model_at_end=True,
    save_total_limit=5,
)

#### viz with tensorboard
#args = TrainingArguments(...,
#    logging_dir="out/tb",
#    report_to=["tensorboard"],
#)
# bash
# tensorboard --logdir out/tb

#### viz with wandb
# args = TrainingArguments(..., report_to=["wandb"], run_name="llm-scratch")

collator = DataCollatorForLanguageModeling(tokenizer, mlm=False) # pad=-100

callbacks = [
    TsvLogger(data_dir / "training.tsv"),
    EarlyStoppingCallback(early_stopping_patience=10, early_stopping_threshold=1e-03),
]

In [9]:
#### 5. setup
model = LlamaForCausalLM(cfg)
#model.resize_token_embeddings(len(tokenizer))
model.config.pad_token_id = tokenizer.pad_token_id

print(f"Memory footprint: {model.get_memory_footprint() / 1e9:.1f} GB")

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=lm_ds["train"],
    eval_dataset=lm_ds["validation"],
    data_collator=collator,
    callbacks=callbacks,
)

#trainer.add_callback(callbacks[0])

Memory footprint: 0.2 GB


In [10]:
#### 5. train
model.config.use_cache = False
model.gradient_checkpointing_enable()

ckpts = sorted(
    data_dir.glob("checkpoint-*/"),
    key=lambda x: int(x.name.replace("checkpoint-", "")),
    reverse=True,
)

if len(ckpts) > 0:
    print(f"==> resume_from_checkpoint: {ckpts[0]}")

trainer.train(resume_from_checkpoint=len(ckpts) > 0)

==> resume_from_checkpoint: data/ch13/checkpoint-1600


Step,Training Loss,Validation Loss
1800,0.882000,9.489492


TrainOutput(global_step=1800, training_loss=0.11153115802341038, metrics={'train_runtime': 228.0727, 'train_samples_per_second': 5612.247, 'train_steps_per_second': 87.691, 'total_flos': 2.0369322657447936e+16, 'train_loss': 0.11153115802341038, 'epoch': 29.527835051546393})

In [11]:
#### 6. save the best model and evaluate loss
print("Best checkpoint:", trainer.state.best_model_checkpoint)
best_ckpt = data_dir / "best"
best_ckpt.mkdir(parents=True, exist_ok=True)

trainer.save_model(best_ckpt)

model.gradient_checkpointing_disable()
model.config.use_cache = True

#eval_loss = trainer.evaluate()["eval_loss"]
#print("PPL:", math.exp(eval_loss))
eval_loss = trainer.evaluate()
print(f"eval_loss: {eval_loss}")

Best checkpoint: data/ch13/checkpoint-600


eval_loss: {'eval_loss': 6.355066299438477, 'eval_runtime': 1.3242, 'eval_samples_per_second': 150.278, 'eval_steps_per_second': 18.879, 'epoch': 29.527835051546393}


In [12]:
#lr_scheduler_type = "linear" | "cosine" | "cosine_with_restarts" | "polynomial" | "constant" | "constant_with_warmup" | "reduce_lr_on_plateau"

#from transformers import Trainer
#from torch.optim.lr_scheduler import MultiStepLR

#class MyTrainer(Trainer):
#    def create_scheduler(self, num_training_steps: int):
#        if self.lr_scheduler is None:
#            self.lr_scheduler = MultiStepLR(self.optimizer, milestones=[1000, 3000, 8000], gamma=0.1)
#        return self.lr_scheduler

#class MyTrainer(Trainer):
#    def create_scheduler(self, num_training_steps: int):
#        if self.lr_scheduler is None:
#            self.lr_scheduler = MultiStepLR(self.optimizer, milestones=[1000, 3000, 8000], gamma=0.1)
#        return self.lr_scheduler

#trainer = MyTrainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds)
#trainer.train()